In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

sns.set_theme(style="whitegrid",font_scale=1.05)

In [2]:
df=pd.read_excel("BLC-19.xlsx")
df.head()

,S.No.,Country,City,Company,Website,Linkedin,Need,Email,Contact,Maps,Category,Rating,Review
0,1,USA,San Francisco,Andrew Dental Clinic San Francisco,NaN,NaN,NaN,NaN,12294541223,https://www.google.com/maps/place/Andrew+Denta...,Dentist,0.0,0
1,2,USA,San Francisco,San Francisco Dental Artistry,NaN,NaN,NaN,NaN,14153990609,https://www.google.com/maps/place/San+Francisc...,Dentist,3.7,74
2,3,India,Haryana,FAMILY DENTAL AND COSMETIC CLINIC,NaN,NaN,NaN,NaN,8708512323,https://www.google.com/maps/place/FAMILY+DENTA...,Hospital,4.9,14
3,4,India,Haryana,Shree Ram Clinic,NaN,NaN,NaN,NaN,7206042912,https://www.google.com/maps/place/Shree+Ram+Cl...,Medical Clinic,5.0,7
4,5,India,Haryana,Sunil's health and wellness physiotherapy clin...,NaN,NaN,NaN,NaN,9992849288,https://www.google.com/maps/place/Sunil's+heal...,Medical Clinic,4.7,11


In [14]:
df["Category"] = df["Category"].str.strip().str.title().fillna("Unknown")


def map_to_standard_category(cat):
    cat_lower = str(cat).lower().strip()

    if 'dental' in cat_lower or 'dentist' in cat_lower or 'orthodont' in cat_lower:
        return 'Dental Clinic'

    if ('eye' in cat_lower or 'optom' in cat_lower or 'optician' in cat_lower
        or 'ophthalm' in cat_lower or 'vision' in cat_lower or 'optical' in cat_lower):
        return 'Eye Care'

    if ('ent specialist' in cat_lower or 'ear, nose' in cat_lower or 'ear nose' in cat_lower
        or 'nose throat' in cat_lower or 'otolaryng' in cat_lower or 'audiolog' in cat_lower
        or 'hearing' in cat_lower or 'hearning' in cat_lower
        or 'speech and hear' in cat_lower or 'speech therap' in cat_lower):
        return 'ENT & Audiology'

    if ('physio' in cat_lower or 'chiropract' in cat_lower
        or 'rehabilit' in cat_lower or 'rehabiliation' in cat_lower
        or 'osteopath' in cat_lower or 'pilates' in cat_lower
        or 'acupuncture' in cat_lower or 'occupational therap' in cat_lower):
        return 'Physiotherapy & Rehab'

    if ('homeopath' in cat_lower or 'homoeopath' in cat_lower
        or 'ayurved' in cat_lower or 'alternative medicine' in cat_lower
        or 'holistic' in cat_lower or 'naturopath' in cat_lower):
        return 'Homeopathy & Alternative'

    if ('pediatric' in cat_lower or 'paediatric' in cat_lower
        or 'child health' in cat_lower or 'baby care' in cat_lower):
        return 'Pediatric Clinic'

    if ('gynecol' in cat_lower or 'gynaecol' in cat_lower
        or 'obstetric' in cat_lower or 'fertility' in cat_lower
        or 'orthopedic' in cat_lower or 'orthopaedic' in cat_lower
        or 'urolog' in cat_lower or 'oncolog' in cat_lower
        or 'dermatolog' in cat_lower or 'skin care' in cat_lower
        or 'neurolog' in cat_lower or 'cardiolog' in cat_lower
        or 'heart' in cat_lower or 'diabet' in cat_lower
        or 'surgeon' in cat_lower or 'radiolog' in cat_lower or 'psychiatr' in cat_lower or 'psycholog' in cat_lower
        or 'mental hospital' in cat_lower or 'mental health' in cat_lower
        or 'eating disorder' in cat_lower or 'behavioral health' in cat_lower
        or 'behavioral medicine' in cat_lower):
        return 'Speciality Clinic'

    if ('hospital' in cat_lower or 'surgical' in cat_lower
        or 'nursing home' in cat_lower or 'trauma' in cat_lower
        or 'urgent care' in cat_lower or 'walk in clinic' in cat_lower or 'ambulance' in cat_lower):
        return 'Hospital'


    if ('medical' in cat_lower or 'clinic' in cat_lower
        or 'doctor' in cat_lower or 'health' in cat_lower
        or 'pharmacy' in cat_lower or 'physician' in cat_lower
        or 'practitioner' in cat_lower or 'home care' in cat_lower
        or 'wellness' in cat_lower or 'addiction' in cat_lower
        or 'general practitioner' in cat_lower or 'nutritionist' in cat_lower or 'diagnostic' in cat_lower or 'imaging' in cat_lower
        or 'x-ray' in cat_lower or 'laborator' in cat_lower
        or 'sonograph' in cat_lower or ' lab' in cat_lower
        or cat_lower == 'lab' or 'veterinar' in cat_lower or 'animal hospital' in cat_lower
        or 'pet clinic' in cat_lower):
        return 'Medical Clinic'

    if ('gym' in cat_lower or 'weight loss' in cat_lower
        or 'fitness' in cat_lower or 'beauty parlour' in cat_lower or 'beauty parlor' in cat_lower
        or 'salon' in cat_lower or 'make-up' in cat_lower
        or 'makeup' in cat_lower or 'massage' in cat_lower):
        return 'Fitness & Wellness'



    if ('restaurant' in cat_lower or 'dhaba' in cat_lower
        or 'cafe' in cat_lower or 'coffee shop' in cat_lower
        or 'bakery' in cat_lower or 'cake shop' in cat_lower
        or 'food court' in cat_lower or 'sweets shop' in cat_lower
        or 'chicken and mutton' in cat_lower or 'dairy' in cat_lower):
        return 'Food & Beverage'


    if ('clothing store' in cat_lower or 'jewelry' in cat_lower
        or 'gift shop' in cat_lower or 'book store' in cat_lower
        or 'stationery' in cat_lower or 'baby store' in cat_lower
        or 'furniture' in cat_lower or 'auto care' in cat_lower
        or 'banking supply' in cat_lower or 'wholesaler' in cat_lower
        or 'ice cream shop' in cat_lower or cat_lower == 'store'
        or 'mattress' in cat_lower):
        return 'Retail Store'

    if ('coaching' in cat_lower or 'tutor' in cat_lower
        or 'tuition' in cat_lower):
        return 'Training Center'

    if ('driving school' in cat_lower
        or 'nursing school' in cat_lower
        or 'college' in cat_lower
        or 'engineering school' in cat_lower
        or 'computer training' in cat_lower
        or 'software training' in cat_lower
        or 'special education' in cat_lower
        or 'preschool' in cat_lower
        or 'kindergar' in cat_lower
        or 'play school' in cat_lower
        or 'play group' in cat_lower
        or 'daycare' in cat_lower
        or 'primary school' in cat_lower
        or 'school' in cat_lower
        or 'academy' in cat_lower
        or 'education' in cat_lower
        or 'library' in cat_lower
        or 'learning center' in cat_lower
        or 'student union' in cat_lower
        or 'music' in cat_lower
        or 'dance' in cat_lower
        or 'art school' in cat_lower
        or 'art studio' in cat_lower
        or 'art center' in cat_lower
        or 'martial art' in cat_lower
        or 'training' in cat_lower
        or 'institute' in cat_lower):
        return 'Training Center'

    if ('business center' in cat_lower or 'apartment complex' in cat_lower
        or 'school administration' in cat_lower
        or 'professional center' in cat_lower):
        return 'Other'

    return 'Other'


df['Standard_Category'] = df['Category'].apply(map_to_standard_category)

category_ranking = df['Standard_Category'].value_counts()

print(category_ranking)

Standard_Category
Physiotherapy & Rehab       41
Training Center             38
Hospital                    25
Medical Clinic              24
Dental Clinic               20
Eye Care                    14
Speciality Clinic           14
ENT & Audiology             10
Fitness & Wellness           8
Retail Store                 2
Homeopathy & Alternative     2
Pediatric Clinic             1
Other                        1
Name: count, dtype: int64


In [16]:
india_ranks = {
    'Training Center': 1,
    'ENT & Audiology': 2,
    'Physiotherapy & Rehab': 3,
    'Medical Clinic': 4,
    'Homeopathy & Alternative': 5,
    'Dental Clinic': 6,
    'Eye Care': 7,
    'Speciality Clinic': 8,
    'Pediatric Clinic': 9,
    'Hospital': 10,
    'Fitness & Wellness': 11,
    'Retail Store': 12,
    'Other': 13
}

foreign_ranks = {
    'Medical Clinic': 1,
    'Hospital': 2,
    'Speciality Clinic': 3,
    'Homeopathy & Alternative': 4,
    'Dental Clinic': 5,
    'Physiotherapy & Rehab': 6,
    'Eye Care': 7,
    'ENT & Audiology': 8,
    'Training Center': 9,
    'Fitness & Wellness': 10,
    'Pediatric Clinic': 11,
    'Retail Store': 12,
    'Other': 13
}
def assign_lead_score_and_rank(row):
    country = str(row['Country']).strip().lower()
    cat = row['Standard_Category']

    if 'india' in country:
        rank = india_ranks.get(cat, 11)
        if rank <= 6: return pd.Series([0.85, rank])
        elif rank <= 10: return pd.Series([0.55, rank])
        else: return pd.Series([0.15, rank])
    else:
        rank = foreign_ranks.get(cat, 11)
        if rank <= 5: return pd.Series([0.85, rank])
        elif rank <= 9: return pd.Series([0.55, rank])
        else: return pd.Series([0.15, rank])

df[['Conversion_Probability', 'Rank']] = df.apply(assign_lead_score_and_rank, axis=1)
print(df[['Country', 'Standard_Category', 'Rank', 'Conversion_Probability']].head(20))

   Country Standard_Category  Rank  Conversion_Probability
0      USA     Dental Clinic   5.0                    0.85
1      USA     Dental Clinic   5.0                    0.85
2    India          Hospital  10.0                    0.55
3    India    Medical Clinic   4.0                    0.85
4    India    Medical Clinic   4.0                    0.85
5    India    Medical Clinic   4.0                    0.85
6    India    Medical Clinic   4.0                    0.85
7    India     Dental Clinic   6.0                    0.85
8    India          Hospital  10.0                    0.55
9    India     Dental Clinic   6.0                    0.85
10   India     Dental Clinic   6.0                    0.85
11   India     Dental Clinic   6.0                    0.85
12   India     Dental Clinic   6.0                    0.85
13   India     Dental Clinic   6.0                    0.85
14   India     Dental Clinic   6.0                    0.85
15   India     Dental Clinic   6.0                    0.

In [17]:
df['Rating'] = pd.to_numeric(df['Rating'], errors='coerce').fillna(0.0)
df['Review'] = pd.to_numeric(df['Review'], errors='coerce').fillna(0.0)

df["Company_Name_Length"] = df["Company"].str.len()
df["Company_Word_Count"] = df["Company"].str.split().str.len()

category_counts_dict = df["Standard_Category"].value_counts().to_dict()
df["Category_Frequency"] = df["Standard_Category"].map(category_counts_dict)

City_counts_dict = df["City"].value_counts().to_dict()
df["City_Frequency"] = df["City"].map(City_counts_dict)

le_city = LabelEncoder()
df["City_Encoded"] = le_city.fit_transform(df["City"])

le_category = LabelEncoder()
df["Category_Encoded"] = le_category.fit_transform(df["Standard_Category"])

print("Features created!")
print(df[["Company_Name_Length", "Company_Word_Count", "Category_Frequency", "City_Frequency"]].head())

Features created!
   Company_Name_Length  Company_Word_Count  Category_Frequency  City_Frequency
0                   34                   5                  20              12
1                   29                   4                  20              12
2                   33                   5                  25              68
3                   16                   3                  24              68
4                   56                   7                  24              68


In [18]:
City_data = df["City"].value_counts().reset_index()
City_data.columns = ["City", "Count"]
City_data = City_data.sort_values(by="Count", ascending=False).head(15)

fig1, ax1 = plt.subplots(figsize=(10, 6))
sns.barplot(data=City_data, x="Count", y="City", palette="crest", ax=ax1, edgecolor="black", linewidth=0.6)
ax1.set_title("Distribution of Businesses by City (Top 15)", fontsize=14, fontweight="bold", pad=15)
ax1.set_xlabel("Number of Businesses", fontsize=12, fontweight="bold")
ax1.set_ylabel("City", fontsize=12, fontweight="bold")
ax1.spines[['top', 'right']].set_visible(False)

for i, v in enumerate(City_data["Count"]):
    ax1.text(v + 0.3, i, str(v), va='center', fontsize=10, fontweight='bold')
plt.tight_layout()
plt.savefig("distribution_of_businesses_by_City.png")
plt.close()

category_counts = df["Standard_Category"].value_counts().reset_index()
category_counts.columns = ["Standard_Category", "Count"]
category_counts = category_counts.sort_values(by="Count", ascending=False)

fig2, ax2 = plt.subplots(figsize=(12, 6))
sns.barplot(data=category_counts, x="Count", y="Standard_Category", palette="viridis", ax=ax2, edgecolor="black", linewidth=0.6)
ax2.set_title("Business Categories by Total Count", fontsize=15, fontweight="bold", pad=15)
ax2.set_xlabel("Number of Businesses", fontsize=12, fontweight="bold")
ax2.set_ylabel("Business Category", fontsize=12, fontweight="bold")
ax2.spines[['top', 'right']].set_visible(False)

for i, v in enumerate(category_counts["Count"]):
    ax2.text(v + 0.3, i, str(v), va='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig("top_business_categories_by_count.png", dpi=300)
plt.close()

print("Graphs generated successfully.")

/tmp/ipykernel_1908/3998404181.py:6: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=City_data, x="Count", y="City", palette="crest", ax=ax1, edgecolor="black", linewidth=0.6)
/tmp/ipykernel_1908/3998404181.py:23: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=category_counts, x="Count", y="Standard_Category", palette="viridis", ax=ax2, edgecolor="black", linewidth=0.6)


Graphs generated successfully.


In [24]:
group_counts = df.groupby(['Country', 'Standard_Category']).size()
df['Group_Size'] = df.apply(lambda row: group_counts[(row['Country'], row['Standard_Category'])], axis=1)
df['Sample_Weight'] = 1 / df['Group_Size']

feature_names = ["Company_Name_Length", "Company_Word_Count",
                 "Category_Frequency", "City_Frequency",
                 "City_Encoded", "Category_Encoded"]
mask = df['Group_Size'] >= 2
df_for_split = df[mask]
df_excluded = df[~mask]

X_split = df_for_split[feature_names]
y_split = df_for_split["Conversion_Probability"]
w_split = df_for_split['Sample_Weight']
strat_key = df_for_split['Country'] + "_" + df_for_split['Standard_Category']

X_train, X_test, y_train, y_test, w_train, w_test = train_test_split(
    X_split, y_split, w_split, test_size=0.2, random_state=42, stratify=strat_key
)

xgb_model = XGBRegressor(n_estimators=700, max_depth=7, random_state=42, learning_rate=0.1)
xgb_model.fit(X_train, y_train, sample_weight=w_train)

print(f"Train R²: {xgb_model.score(X_train, y_train):.4f}")
print(f"Test R²: {xgb_model.score(X_test, y_test):.4f}")
df["Predicted_Probability"] = xgb_model.predict(df[feature_names])

Train R²: 0.9967
Test R²: 0.9853


In [25]:
category_rankings = df.groupby("Standard_Category")["Predicted_Probability"].mean().reset_index()
category_rankings = category_rankings.sort_values(by="Predicted_Probability", ascending=False)

fig3, ax3 = plt.subplots(figsize=(12, 6))
sns.barplot(data=category_rankings, x="Predicted_Probability", y="Standard_Category", palette="viridis", ax=ax3)
ax3.set_title("Business Categories by Predicted Conversion Probability")
ax3.set_xlabel("Average Predicted Probability")
ax3.set_ylabel("Business Category")
plt.tight_layout()
plt.savefig("top_business_categories_by_probability.png")
plt.close()


country_counts = df["Country"].value_counts().reset_index()
country_counts.columns = ["Country", "Count"]

fig4, ax4 = plt.subplots(figsize=(8, 5))
sns.barplot(data=country_counts, x="Count",y="Country",palette="coolwarm",ax=ax4)
ax4.set_title("Lead Volume by Country")
ax4.set_xlabel("Number of Businesses")
ax4.set_ylabel("Country")
plt.tight_layout()
plt.savefig("lead_volume_by_country.png")
plt.close()

/tmp/ipykernel_1908/1707134114.py:5: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=category_rankings, x="Predicted_Probability", y="Standard_Category", palette="viridis", ax=ax3)
/tmp/ipykernel_1908/1707134114.py:18: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=country_counts, x="Count",y="Country",palette="coolwarm",ax=ax4)


In [26]:
columns_to_export = ["S.No.", "Country", "City", "Company", "Contact", "Maps", "Category",
                      "Standard_Category", "Rank", "Predicted_Probability"]

indian_df = df[df["Country"].str.lower().str.contains("india")].sort_values(by="Rank", ascending=True)[columns_to_export]
foreign_df = df[~df["Country"].str.lower().str.contains("india")].sort_values(by="Rank", ascending=True)[columns_to_export]

wb = openpyxl.Workbook()
wb.remove(wb.active)

sheets_config = {
    f"Indian Businesses ({len(indian_df)})": (indian_df, "1F4E78"),
    f"Foreign Businesses ({len(foreign_df)})": (foreign_df, "4D4D4D")
}

for sheet_name, (sub_df, header_color) in sheets_config.items():
    ws = wb.create_sheet(title=sheet_name)
    ws.views.sheetView[0].showGridLines = False

    headers = list(sub_df.columns)
    ws.append(headers)

    header_font = Font(name="Calibri", size=11, bold=True, color="FFFFFF")
    header_fill = PatternFill(start_color=header_color, end_color=header_color, fill_type="solid")
    center_align = Alignment(horizontal="center", vertical="center", wrap_text=True)
    left_align = Alignment(horizontal="left", vertical="center")
    thin_border = Border(
        left=Side(style="thin", color="D9D9D9"), right=Side(style="thin", color="D9D9D9"),
        top=Side(style="thin", color="D9D9D9"), bottom=Side(style="thin", color="D9D9D9")
    )
    row_fill_even = PatternFill(start_color="F2F6FA", end_color="F2F6FA", fill_type="solid")
    row_fill_odd = PatternFill(start_color="FFFFFF", end_color="FFFFFF", fill_type="solid")

    for col_num, header in enumerate(headers, 1):
        cell = ws.cell(row=1, column=col_num)
        cell.font = header_font
        cell.fill = header_fill
        cell.alignment = center_align
        cell.border = thin_border
    ws.row_dimensions[1].height = 28

    for r_idx, row_data in enumerate(sub_df.values, 2):
        row_fill = row_fill_even if r_idx % 2 == 0 else row_fill_odd
        for c_idx, value in enumerate(row_data, 1):
            current_header = headers[c_idx - 1]

            if current_header == "Contact":
                clean_contact = str(value).strip() if pd.notna(value) else ""
                clean_contact = "" if clean_contact.lower() == "nan" else clean_contact
                cell = ws.cell(row=r_idx, column=c_idx)
                cell.value = "'" + clean_contact if clean_contact else ""
                cell.data_type = "s"
                cell.number_format = "@"
                cell.alignment = center_align
                cell.border = thin_border
                cell.font = Font(name="Calibri", size=10.5)
                cell.fill = row_fill
                continue

            cell = ws.cell(row=r_idx, column=c_idx, value=value)
            cell.border = thin_border
            cell.font = Font(name="Calibri", size=10.5)
            cell.fill = row_fill

            if current_header in ["S.No", "Rank"]:
                cell.alignment = center_align
            elif current_header == "Predicted_Probability":
                cell.alignment = center_align
                cell.number_format = "0.0%"
                if isinstance(value, (int, float)):
                    if value >= 0.70:
                        cell.font = Font(name="Calibri", size=10.5, bold=True, color="1E7C3A")
                    elif value <= 0.30:
                        cell.font = Font(name="Calibri", size=10.5, color="A6A6A6")
            elif current_header == "Maps":
                cell.alignment = left_align
                if pd.notna(value) and str(value).startswith("http"):
                    cell.hyperlink = str(value)
                    cell.font = Font(name="Calibri", size=10.5, color="0563C1", underline="single", bold=True)
                else:
                    cell.value = "N/A"
                    cell.font = Font(name="Calibri", size=10.5, color="A6A6A6")
            else:
                cell.alignment = left_align

        ws.row_dimensions[r_idx].height = 20

    for col in ws.columns:
        max_len = max(len(str(cell.value or "")) for cell in col)
        col_letter = get_column_letter(col[0].column)
        ws.column_dimensions[col_letter].width = max(max_len + 3, 12)

    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions

wb.save("leads_prioritized_by_country.xlsx")

print("\n--- PROCESS COMPLETE ---")
print("Successfully generated 'leads_prioritized_by_country.xlsx' with Indian and Foreign sheets sorted by Rank!")


--- PROCESS COMPLETE ---
Successfully generated 'leads_prioritized_by_country.xlsx' with Indian and Foreign sheets sorted by Rank!


In [27]:
df['Contact'] = df['Contact'].astype(str).str.strip()
df['Contact'] = df['Contact'].apply(lambda x: "'" + x if x not in ['nan', 'NaN', '', 'None'] else '')
columns_for_second_notebook = [
    "S.No.", "Country", "City", "Company", "Contact", "Maps", "Category",
    "Standard_Category", "Rank", "Conversion_Probability",
    "Predicted_Probability", "Rating", "Review"
]
df[columns_for_second_notebook].to_csv("blc_cleaned.csv", index=False)
print("Clean dataset exported successfully as 'blc_cleaned.csv'")
print(f"Total records: {len(df)}")

Clean dataset exported successfully as 'blc_cleaned.csv'
Total records: 200
